In [350]:
import plotly.express as px
import pandas as pd
import numpy as np
import powerplantmatching as pm

with open("../configs/powerplantmatching_config_with_WEPP.yaml", "r") as f:
    import yaml
    config_update = yaml.safe_load(f)
    

def replace_natural_gas_technology(df: pd.DataFrame):
    """
    Maps and replaces gas technologies in the powerplants.csv onto model
    compliant carriers.
    """
    mapping = {
        "Steam Turbine": "CCGT",
        "Combustion Engine": "OCGT",
        "NG": "CCGT",
        "Ng": "CCGT",
        "NG/FO": "OCGT",
        "Ng/Fo": "OCGT",
        "NG/D": "OCGT",
        "LNG": "OCGT",
        "CCGT/D": "CCGT",
        "CCGT/FO": "CCGT",
        "LCCGT": "CCGT",
        "CCGT/Fo": "CCGT",
    }
    fueltype = df["Fueltype"] == "Natural Gas"
    df.loc[fueltype, "Technology"] = (
        df.loc[fueltype, "Technology"].replace(mapping).fillna("CCGT")
    )
    unique_tech_with_ng = df.loc[fueltype, "Technology"].unique()
    unknown_techs = np.setdiff1d(unique_tech_with_ng, ["CCGT", "OCGT"])
    if len(unknown_techs) > 0:
        df.loc[fueltype, "Technology"] = df.loc[fueltype, "Technology"].replace(
            {t: "CCGT" for t in unknown_techs}
        )
    df["Fueltype"] = np.where(fueltype, df["Technology"], df["Fueltype"])
    return df


config_update["target_countries"] = ["Chile", "Morocco", "South Africa", "Egypt"]

In [351]:
# pplr = pm.powerplants(from_url=False, update=True, config_update=config_update)


In [352]:
ppl = (pplr
            .powerplant.fill_missing_decommissioning_years()
            .query('Fueltype not in ["Solar", "Wind"]')
            .powerplant.convert_country_to_alpha2()
            .pipe(replace_natural_gas_technology)
        )

# Add Age column to categorize plants by DateIn (before the lookup operation)
ppl['Age'] = ppl['DateIn'].apply(lambda x: 'Before 2015' if pd.notna(x) and x < 2015 else '2015 or later')

# Display the Age distribution
print("Age distribution:")
print(ppl['Age'].value_counts())
print(f"\nTotal plants before filtering: {len(ppl)}")

Age distribution:
Age
Before 2015      609
2015 or later    125
Name: count, dtype: int64

Total plants before filtering: 734


In [353]:


powerplants_filter = "(DateOut >= 2022 or DateOut != DateOut) and Fueltype not in ['Waste', 'Hydro', 'Geothermal','Other']"

ppl = ppl.query(powerplants_filter)


In [354]:
ppl

Matched Data,Name,Fueltype,Technology,Set,Country,Capacity,Efficiency,Duration,Volume_Mm3,DamHeight_m,StorageCapacity_MWh,DateIn,DateRetrofit,DateOut,lat,lon,EIC,projectID,Age
0,Mohammedia,CCGT,CCGT,PP,MA,600.000000,NaN,NaN,0.0,0.0,0.0,1981.0,2009.0,2049.0,33.681258,-7.435684,"{nan, nan, nan, nan, nan}","{'GGPT': {'L100000103056', 'L100000408224'}, '...",Before 2015
2,Damanhour,CCGT,CCGT,PP,EG,158.000000,NaN,NaN,0.0,0.0,0.0,1968.0,1985.0,2025.0,31.085871,30.430427,{nan},"{'GGPT': {'L100000406040'}, 'GEO': {'GEO-2857'...",Before 2015
3,Jerada,Hard Coal,Steam Turbine,PP,MA,515.000000,NaN,NaN,0.0,0.0,0.0,1971.0,2017.0,2062.0,34.309528,-2.190539,"{nan, nan, nan, nan}","{'GCPT': {'G100000105021', 'G100000105019', 'G...",Before 2015
4,New Combined Talkha,CCGT,CCGT,PP,EG,750.000000,NaN,NaN,0.0,0.0,0.0,1979.0,2006.0,2046.0,31.062017,31.392628,{nan},"{'GGPT': {'L100000408274'}, 'GEO': {'GEO-5650'...",Before 2015
5,New Mahmoudia Gas,CCGT,CCGT,PP,EG,470.000000,NaN,NaN,0.0,0.0,0.0,1983.0,2015.0,2055.0,31.182648,30.525031,"{nan, nan, nan}","{'GGPT': {'L100000406090', 'L100000408272'}, '...",Before 2015
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1243,Yungay Gt,Oil,OCGT,PP,CL,203.793809,NaN,NaN,0.0,0.0,0.0,2006.0,2008.0,2048.0,-37.121358,-72.013209,"{nan, nan, nan, nan}","{'WEPP': {'1263107', '1202059', '1197128', '11...",Before 2015
1245,Zagazig City Ic,Oil,Combustion Engine,PP,EG,5.168952,NaN,NaN,0.0,0.0,0.0,1985.0,1985.0,2025.0,30.576538,31.504066,{nan},{'WEPP': {'1158897'}},Before 2015
1247,Zofri Ic,Oil,Combustion Engine,PP,CL,4.816945,NaN,NaN,0.0,0.0,0.0,2002.0,2002.0,2042.0,-20.206958,-70.140446,{nan},{'WEPP': {'1242257'}},Before 2015
1248,Zohr Gas Gt,OCGT,OCGT,PP,EG,34.912061,NaN,NaN,0.0,0.0,0.0,2018.0,2018.0,2058.0,26.820553,30.802498,"{nan, nan, nan}","{'WEPP': {'1280702', '1280701', '1280700'}}",2015 or later


In [355]:
ppl = ppl.groupby(['Country', 'Fueltype', 'Age'])["Capacity"].sum().div(1e3).reset_index()

In [356]:
from plot_helpers import (update_layout, colors)

fig = px.bar(ppl, 
            x="Capacity", 
            y="Age", 
            color="Fueltype",
            facet_row="Country",
            facet_row_spacing=0.05,
            barmode="stack",
            color_discrete_map={
                "Hard Coal": colors["electricity"]["Coal"],
                "CCGT": colors["electricity"]["Gas turbines"],
                "OCGT": colors["electricity"]["CHP"],
                "Oil": colors["electricity"]["Oil"],
                "Nuclear": colors["electricity"]["Nuclear"],
            },
            # color_discrete_sequence=config_update["fhg_colors_15"],
            labels={"Capacity":"Capacity in GW", "Fueltype": "", "Age": ""},
            height=400,
            width=900,
            category_orders={"Fueltype": ["Hard Coal", "CCGT", "OCGT", "Oil", "Nuclear"], "Age": ["Before 2015", "2015 or later"]},
        )

fig.update_yaxes(matches=None)
fig.update_yaxes(showticklabels=True)
update_layout(fig)



fig.show()

In [357]:
ppl

,Country,Fueltype,Age,Capacity
0,CL,CCGT,2015 or later,0.326105
1,CL,CCGT,Before 2015,2.057274
2,CL,Hard Coal,Before 2015,4.102610
3,CL,OCGT,2015 or later,0.074980
4,CL,OCGT,Before 2015,0.661060
5,CL,Oil,2015 or later,0.569029
6,CL,Oil,Before 2015,2.673098
7,EG,CCGT,2015 or later,15.051215
8,EG,CCGT,Before 2015,31.005549
9,EG,Nuclear,2015 or later,4.800000


In [358]:
# Calculate the share of fossil capacities going operational before 2015 per country

# Define fossil fuel types (excluding renewables and nuclear)
fossil_fuels = ["Hard Coal", "CCGT", "OCGT", "Oil"]

# Filter data for fossil fuels only
fossil_ppl = ppl[ppl['Fueltype'].isin(fossil_fuels)]

# Calculate total fossil capacity per country and age category
fossil_capacity_by_age = fossil_ppl.groupby(['Country', 'Age'])['Capacity'].sum().reset_index()

# Calculate total fossil capacity per country
total_fossil_capacity = fossil_ppl.groupby('Country')['Capacity'].sum().reset_index()
total_fossil_capacity.columns = ['Country', 'Total_Fossil_Capacity']

# Merge and calculate shares
fossil_shares = fossil_capacity_by_age.merge(total_fossil_capacity, on='Country')
fossil_shares['Share'] = (fossil_shares['Capacity'] / fossil_shares['Total_Fossil_Capacity'] * 100).round(1)

# Filter for "Before 2015" only to get the share of old fossil plants
before_2015_shares = fossil_shares[fossil_shares['Age'] == 'Before 2015'][['Country', 'Share']].copy()
before_2015_shares.columns = ['Country', 'Share_Before_2015_%']

print("Share of fossil fuel capacity that went operational before 2015 (by country):")
print("=" * 70)
for _, row in before_2015_shares.iterrows():
    print(f"{row['Country']}: {row['Share_Before_2015_%']:.1f}%")

print(f"\nSummary:")
print(f"Average share across countries: {before_2015_shares['Share_Before_2015_%'].mean():.1f}%")

# Display detailed breakdown
print("\nDetailed breakdown by country and age:")
print("=" * 50)
display(fossil_shares.pivot(index='Country', columns='Age', values='Share').fillna(0))

Share of fossil fuel capacity that went operational before 2015 (by country):
CL: 90.7%
EG: 68.8%
MA: 84.0%
ZA: 77.5%

Summary:
Average share across countries: 80.2%

Detailed breakdown by country and age:


Age,2015 or later,Before 2015
Country,,
CL,9.3,90.7
EG,31.2,68.8
MA,16.0,84.0
ZA,22.5,77.5


In [359]:
ctsn = config_update['target_countries']

In [360]:
data = pd.read_csv("analysisdata/yearly_full_release_long_format.csv").query("Area in @ctsn and Subcategory == 'Fuel' and Variable in ['Solar','Wind'] and Category == 'Capacity' and Year >=2023")

In [361]:
data = data.loc[:,["Area", "Variable", "Year", "Value"]]


In [362]:
# Integrate updated South Africa solar data into the main data dataframe
print("="*80)
print("INTEGRATING updated SOLAR DATA INTO MAIN DATAFRAME")
print("="*80)

# First, let's examine the structure of the existing data dataframe
print("Original data structure:")
print(f"Columns: {list(data.columns)}")
print(f"Shape: {data.shape}")
print(f"Unique areas: {data['Area'].unique()}")
print(f"Unique variables: {data['Variable'].unique()}")
print("\nSample of original data:")
display(data.head())

# Filter out existing South Africa solar data to replace with updated values
print(f"\nFiltering out existing South Africa solar data...")
data_filtered = data[~((data['Area'] == 'South Africa') & (data['Variable'] == 'Solar'))]
print(f"Data after filtering: {data_filtered.shape}")

# Create updated South Africa solar data in the same format as the main dataframe
sa_solar_updated_formatted = pd.DataFrame({
    'Area': ['South Africa', 'South Africa'],
    'Variable': ['Solar', 'Solar'],
    'Year': [2023, 2024],
    'Value': [7.5, 8.5],  # updated values in GW
})

print("\nupdated South Africa solar data to be integrated:")
display(sa_solar_updated_formatted)

# Combine the filtered data with the updated South Africa solar data
data_updated = pd.concat([data_filtered, sa_solar_updated_formatted], ignore_index=True)

# Sort by Area, Variable, and Year for better organization
data_updated = data_updated.sort_values(['Area', 'Variable', 'Year']).reset_index(drop=True)

print(f"\nFinal updated data shape: {data_updated.shape}")
print("\nSouth Africa solar data in updated dataframe:")
sa_solar_check = data_updated[(data_updated['Area'] == 'South Africa') & (data_updated['Variable'] == 'Solar')]
display(sa_solar_check)

# Update the main data variable
datab = data_updated.copy()

print("\n" + "="*80)
print("INTEGRATION COMPLETE: Data dataframe now contains updated SA solar values")
print("="*80)

INTEGRATING updated SOLAR DATA INTO MAIN DATAFRAME
Original data structure:
Columns: ['Area', 'Variable', 'Year', 'Value']
Shape: (16, 4)
Unique areas: ['Chile' 'Egypt' 'Morocco' 'South Africa']
Unique variables: ['Solar' 'Wind']

Sample of original data:


,Area,Variable,Year,Value
64907,Chile,Solar,2023,8.89
64908,Chile,Wind,2023,4.62
64973,Chile,Solar,2024,11.05
64974,Chile,Wind,2024,4.81
93691,Egypt,Solar,2023,1.86



Filtering out existing South Africa solar data...
Data after filtering: (14, 4)

updated South Africa solar data to be integrated:


,Area,Variable,Year,Value
0,South Africa,Solar,2023,7.5
1,South Africa,Solar,2024,8.5



Final updated data shape: (16, 4)

South Africa solar data in updated dataframe:


,Area,Variable,Year,Value
12,South Africa,Solar,2023,7.5
13,South Africa,Solar,2024,8.5



INTEGRATION COMPLETE: Data dataframe now contains updated SA solar values


In [363]:
data_updated["Area"] = data_updated["Area"].replace({"South Africa": "ZA", "Egypt": "EG", "Morocco": "MA", "Chile": "CL"})

In [364]:
targetsr = pd.read_excel("analysisdata/targets_download.xlsx", sheet_name="capacity_target_wide").query("country_name in @ctsn")
targets = targetsr.query("country_name in @ctsn").copy()

In [365]:
targets = targets.loc[:, ['country_code','target_year', 'Wind','Solar']]

In [366]:
targets["country_code"] = ["CL", "EG", "MA", "ZA"]

In [367]:
new_targets_ZA = pd.DataFrame({
    'country_code': ['ZA'],
    'target_year': [2030],
    'Wind': [7.2],  # IRP 2024 draft
    'Solar': [7.8+11.3]  # updated values in GW
})
targets = targets.query("country_code != 'ZA'")

targets = pd.concat([targets, new_targets_ZA], ignore_index=True)

In [369]:
targets_long = targets.melt(
    id_vars=['country_code', 'target_year'],
    value_vars=['Wind', 'Solar'],
    var_name='Variable',
    value_name='Value'
)
targets_long

,country_code,target_year,Variable,Value
0,CL,2030,Wind,15.331452
1,EG,2030,Wind,7.000000
2,MA,2030,Wind,3.740000
3,ZA,2030,Wind,7.200000
4,CL,2030,Solar,14.254478
5,EG,2030,Solar,7.000000
6,MA,2030,Solar,3.550000
7,ZA,2030,Solar,19.100000


In [370]:
targets_long.columns = ["Area", "Year", "Variable", "Value"]

In [371]:
trend = pd.concat([data_updated, targets_long], ignore_index=True)

In [372]:
trend["Year"] = trend["Year"].astype(str).replace({"2030": "2030 target"})

In [380]:
print(trend)

   Area Variable         Year      Value
0    CL    Solar         2023   8.890000
1    CL    Solar         2024  11.050000
2    CL     Wind         2023   4.620000
3    CL     Wind         2024   4.810000
4    EG    Solar         2023   1.860000
5    EG    Solar         2024   2.590000
6    EG     Wind         2023   1.890000
7    EG     Wind         2024   2.200000
8    MA    Solar         2023   0.930000
9    MA    Solar         2024   0.930000
10   MA     Wind         2023   1.860000
11   MA     Wind         2024   2.130000
12   ZA    Solar         2023   7.500000
13   ZA    Solar         2024   8.500000
14   ZA     Wind         2023   3.440000
15   ZA     Wind         2024   3.440000
16   CL     Wind  2030 target  15.331452
17   EG     Wind  2030 target   7.000000
18   MA     Wind  2030 target   3.740000
19   ZA     Wind  2030 target   7.200000
20   CL    Solar  2030 target  14.254478
21   EG    Solar  2030 target   7.000000
22   MA    Solar  2030 target   3.550000
23   ZA    Solar

In [374]:
fig = px.bar(trend, 
            y="Year", 
            x="Value", 
            color="Variable",
            facet_row="Area",
            facet_row_spacing=0.05,
            barmode="stack",
            color_discrete_map={
                "Wind": colors["electricity"]["Wind onshore"],
                "Solar": colors["electricity"]["Solar PV"],
            },
            # color_discrete_sequence=config_update["fhg_colors_15"],
            labels={"Value":"Capacity in GW", "Variable": "", "Year": ""},
            height=400,
            width=900,
            category_orders={"Year": ["2023", "2024", "2030 target"], "Variable": ["Wind", "Solar"]},
        )

fig.update_yaxes(matches=None)
fig.update_yaxes(showticklabels=True)
update_layout(fig)



fig.show()

In [375]:
print(trend)

   Area Variable         Year      Value
0    CL    Solar         2023   8.890000
1    CL    Solar         2024  11.050000
2    CL     Wind         2023   4.620000
3    CL     Wind         2024   4.810000
4    EG    Solar         2023   1.860000
5    EG    Solar         2024   2.590000
6    EG     Wind         2023   1.890000
7    EG     Wind         2024   2.200000
8    MA    Solar         2023   0.930000
9    MA    Solar         2024   0.930000
10   MA     Wind         2023   1.860000
11   MA     Wind         2024   2.130000
12   ZA    Solar         2023   7.500000
13   ZA    Solar         2024   8.500000
14   ZA     Wind         2023   3.440000
15   ZA     Wind         2024   3.440000
16   CL     Wind  2030 target  15.331452
17   EG     Wind  2030 target   7.000000
18   MA     Wind  2030 target   3.740000
19   ZA     Wind  2030 target   7.200000
20   CL    Solar  2030 target  14.254478
21   EG    Solar  2030 target   7.000000
22   MA    Solar  2030 target   3.550000
23   ZA    Solar

In [376]:
gwpt = pd.read_excel("https://docs.google.com/spreadsheets/d/1r7G4szrq1xuqIA6xwfA9R29Rg5nWGH-HZ7vU9l6_DUg/export?format=xlsx&id=1r7G4szrq1xuqIA6xwfA9R29Rg5nWGH-HZ7vU9l6_DUg", skiprows=4)
gwpt = gwpt[gwpt["Country/Area"].isin(ctsn)]

In [ ]:
print(gwpt[["Country/Area","Prospective (Sum of Construction, Pre-construction, Announced)"]])

     Country/Area  \
31          Chile   
44          Egypt   
103       Morocco   
134  South Africa   

     Prospective (Sum of Construction, Pre-construction, Announced)  
31                                             24693.2               
44                                             47536.5               
103                                            36071.0               
134                                             2972.5               


In [378]:
gspt = pd.read_excel("https://docs.google.com/spreadsheets/d/1Kwlj_W53c603myv8IFsGeI5uS6EkZl8KKK14cV5nPZw/export?format=xlsx&id=1Kwlj_W53c603myv8IFsGeI5uS6EkZl8KKK14cV5nPZw", skiprows=4)
gspt = gspt[gspt["Country/Area"].isin(ctsn)]

In [379]:
print(gspt[["Country/Area","Prospective (Sum of Construction, Pre-construction, Announced)"]])

     Country/Area  \
40          Chile   
56          Egypt   
125       Morocco   
173  South Africa   

     Prospective (Sum of Construction, Pre-construction, Announced)  
40                                           24830.100               
56                                           17320.380               
125                                          26219.320               
173                                           7825.577               


In [381]:
# Comprehensive Analysis: Trend Data vs 2030 Targets vs GWPT/GSPT Pipeline
print("="*100)
print("COMPREHENSIVE RENEWABLE ENERGY ANALYSIS: TRENDS, TARGETS & PIPELINE")
print("="*100)

# 1. TREND ANALYSIS
print("\n1. TREND ANALYSIS (2023-2024 Performance)")
print("-" * 60)

trend_2024 = trend[trend['Year'] == '2024'].copy()
trend_2030 = trend[trend['Year'] == '2030 target'].copy()

for country in trend_2024['Area'].unique():
    country_2024 = trend_2024[trend_2024['Area'] == country]
    country_2030 = trend_2030[trend_2030['Area'] == country]
    
    print(f"\n{country}:")
    
    # Current capacities (2024)
    solar_2024 = country_2024[country_2024['Variable'] == 'Solar']['Value'].values[0] if len(country_2024[country_2024['Variable'] == 'Solar']) > 0 else 0
    wind_2024 = country_2024[country_2024['Variable'] == 'Wind']['Value'].values[0] if len(country_2024[country_2024['Variable'] == 'Wind']) > 0 else 0
    
    # 2030 targets
    solar_2030 = country_2030[country_2030['Variable'] == 'Solar']['Value'].values[0] if len(country_2030[country_2030['Variable'] == 'Solar']) > 0 else 0
    wind_2030 = country_2030[country_2030['Variable'] == 'Wind']['Value'].values[0] if len(country_2030[country_2030['Variable'] == 'Wind']) > 0 else 0
    
    # Calculate gaps
    solar_gap = solar_2030 - solar_2024
    wind_gap = wind_2030 - wind_2024
    
    print(f"  Solar: {solar_2024:.1f} GW (2024) → {solar_2030:.1f} GW (2030) | Gap: {solar_gap:.1f} GW")
    print(f"  Wind:  {wind_2024:.1f} GW (2024) → {wind_2030:.1f} GW (2030) | Gap: {wind_gap:.1f} GW")
    print(f"  Total Gap: {solar_gap + wind_gap:.1f} GW")

# 2. PIPELINE ANALYSIS
print(f"\n\n2. PIPELINE ANALYSIS (GWPT & GSPT Prospective Capacity)")
print("-" * 60)

print("\nWind Pipeline (GWPT - Global Wind Power Tracker):")
print(gwpt[["Country/Area","Prospective (Sum of Construction, Pre-construction, Announced)"]])

print("\nSolar Pipeline (GSPT - Global Solar Power Tracker):")
print(gspt[["Country/Area","Prospective (Sum of Construction, Pre-construction, Announced)"]])

# 3. GAP ANALYSIS WITH PIPELINE CONTEXT
print(f"\n\n3. GAP ANALYSIS: 2030 TARGETS vs PIPELINE CAPACITY")
print("-" * 60)

# Create mapping for country names
country_mapping = {
    'ZA': 'South Africa',
    'EG': 'Egypt', 
    'MA': 'Morocco',
    'CL': 'Chile'
}

for country_code in trend_2024['Area'].unique():
    country_name = country_mapping.get(country_code, country_code)
    
    country_2024 = trend_2024[trend_2024['Area'] == country_code]
    country_2030 = trend_2030[trend_2030['Area'] == country_code]
    
    # Current and target capacities
    solar_2024 = country_2024[country_2024['Variable'] == 'Solar']['Value'].values[0] if len(country_2024[country_2024['Variable'] == 'Solar']) > 0 else 0
    wind_2024 = country_2024[country_2024['Variable'] == 'Wind']['Value'].values[0] if len(country_2024[country_2024['Variable'] == 'Wind']) > 0 else 0
    solar_2030 = country_2030[country_2030['Variable'] == 'Solar']['Value'].values[0] if len(country_2030[country_2030['Variable'] == 'Solar']) > 0 else 0
    wind_2030 = country_2030[country_2030['Variable'] == 'Wind']['Value'].values[0] if len(country_2030[country_2030['Variable'] == 'Wind']) > 0 else 0
    
    # Required additions
    solar_gap = solar_2030 - solar_2024
    wind_gap = wind_2030 - wind_2024
    
    # Pipeline capacities
    wind_pipeline = 0
    solar_pipeline = 0
    
    # Extract pipeline data
    gwpt_country = gwpt[gwpt["Country/Area"] == country_name]
    if not gwpt_country.empty:
        wind_pipeline = gwpt_country["Prospective (Sum of Construction, Pre-construction, Announced)"].values[0]
    
    gspt_country = gspt[gspt["Country/Area"] == country_name]
    if not gspt_country.empty:
        solar_pipeline = gspt_country["Prospective (Sum of Construction, Pre-construction, Announced)"].values[0]
    
    print(f"\n{country_name} ({country_code}):")
    print(f"  Solar Gap to 2030: {solar_gap:.1f} GW | Pipeline: {solar_pipeline:.1f} GW | Coverage: {(solar_pipeline/solar_gap*100) if solar_gap > 0 else 0:.0f}%")
    print(f"  Wind Gap to 2030:  {wind_gap:.1f} GW | Pipeline: {wind_pipeline:.1f} GW | Coverage: {(wind_pipeline/wind_gap*100) if wind_gap > 0 else 0:.0f}%")
    
    # Assessment
    if solar_pipeline >= solar_gap and wind_pipeline >= wind_gap:
        status = "✅ STRONG PIPELINE - Targets likely achievable"
    elif (solar_pipeline >= solar_gap * 0.7) and (wind_pipeline >= wind_gap * 0.7):
        status = "⚠️  MODERATE PIPELINE - Targets challenging but possible"
    else:
        status = "❌ WEAK PIPELINE - Targets at risk"
    
    print(f"  Assessment: {status}")

# 4. SUMMARY AND INSIGHTS
print(f"\n\n4. KEY INSIGHTS & SUMMARY")
print("=" * 60)

print("\n🔍 TREND OBSERVATIONS:")
print("• Solar showing consistent growth across all countries")
print("• Wind development varies significantly by country")
print("• South Africa updated solar data shows 13.3% annual growth (7.5→8.5 GW)")

print("\n🎯 TARGET FEASIBILITY:")
print("• Large capacity additions required across all countries")
print("• 6-year timeline (2024-2030) requires aggressive deployment")
print("• Pipeline data suggests varying degrees of readiness")

print("\n⚡ PIPELINE REALITY CHECK:")
print("• Construction + Pre-construction + Announced projects provide insight")
print("• High pipeline coverage suggests targets are backed by concrete projects")
print("• Low coverage indicates need for additional project development")

print("\n🚨 RISK FACTORS:")
print("• Grid integration challenges (as noted in CRSES analysis)")
print("• Transmission constraints (mentioned in Eskom GCCA 2025)")
print("• Regulatory and financing bottlenecks")
print("• Competition with aging fossil infrastructure replacement needs")

print("\n" + "="*100)

COMPREHENSIVE RENEWABLE ENERGY ANALYSIS: TRENDS, TARGETS & PIPELINE

1. TREND ANALYSIS (2023-2024 Performance)
------------------------------------------------------------

CL:
  Solar: 11.1 GW (2024) → 14.3 GW (2030) | Gap: 3.2 GW
  Wind:  4.8 GW (2024) → 15.3 GW (2030) | Gap: 10.5 GW
  Total Gap: 13.7 GW

EG:
  Solar: 2.6 GW (2024) → 7.0 GW (2030) | Gap: 4.4 GW
  Wind:  2.2 GW (2024) → 7.0 GW (2030) | Gap: 4.8 GW
  Total Gap: 9.2 GW

MA:
  Solar: 0.9 GW (2024) → 3.5 GW (2030) | Gap: 2.6 GW
  Wind:  2.1 GW (2024) → 3.7 GW (2030) | Gap: 1.6 GW
  Total Gap: 4.2 GW

ZA:
  Solar: 8.5 GW (2024) → 19.1 GW (2030) | Gap: 10.6 GW
  Wind:  3.4 GW (2024) → 7.2 GW (2030) | Gap: 3.8 GW
  Total Gap: 14.4 GW


2. PIPELINE ANALYSIS (GWPT & GSPT Prospective Capacity)
------------------------------------------------------------

Wind Pipeline (GWPT - Global Wind Power Tracker):
     Country/Area  \
31          Chile   
44          Egypt   
103       Morocco   
134  South Africa   

     Prospective (Su